# USD/LKR Importer Risk Simulator: prepared data

This notebook inspects the Stage 2 data products. It does not fit GBM or run a simulation. From the repository root, run python -m src.prepare_data first if the processed files need refreshing.

In [ ]:
from pathlib import Path
import json
import sys

REPOSITORY_NAME = "usd-lkr-importer-risk-simulator"
candidates = [Path.cwd(), Path.cwd() / REPOSITORY_NAME, Path("/content") / REPOSITORY_NAME]
repo_root = next((path.resolve() for path in candidates if (path / "src" / "prepare_data.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the repository root, or clone the repository in Colab first.")
sys.path.insert(0, str(repo_root))
data_dir = repo_root / "data" / "processed"
print(f"Repository: {repo_root}")
print(f"Processed data: {data_dir}")

## Read the audit and period products

The report and CSV files are generated by the repository pipeline. This cell reads them without modifying the raw or processed data.

In [ ]:
import pandas as pd

report = json.loads((data_dir / "data_quality_report.json").read_text(encoding="utf-8"))
quotes = pd.read_csv(data_dir / "cleaned_usd_lkr.csv", parse_dates=["date"])
returns = pd.read_csv(data_dir / "daily_log_returns.csv", parse_dates=["previous_date", "date"])
calibration_quotes = pd.read_csv(data_dir / "calibration_quotes_2019_2024.csv", parse_dates=["date"])
evaluation_quotes = pd.read_csv(data_dir / "evaluation_quotes_2025_2026.csv", parse_dates=["date"])
calibration_returns = pd.read_csv(data_dir / "calibration_returns_2019_2024.csv", parse_dates=["previous_date", "date"])
evaluation_returns = pd.read_csv(data_dir / "evaluation_returns_2025_2026.csv", parse_dates=["previous_date", "date"])

pd.DataFrame([
    {"series": "All quotes", "rows": len(quotes), "first date": quotes["date"].min(), "last date": quotes["date"].max()},
    {"series": "Calibration quotes", "rows": len(calibration_quotes), "first date": calibration_quotes["date"].min(), "last date": calibration_quotes["date"].max()},
    {"series": "Evaluation quotes", "rows": len(evaluation_quotes), "first date": evaluation_quotes["date"].min(), "last date": evaluation_quotes["date"].max()},
])

## Inspect returns and the selected time intervals

Returns are calculated between adjacent observed selling-rate quotes. calendar_gap_days exposes intervals longer than one calendar day so a later calibration can choose and document a time basis.

In [ ]:
print("Source snapshot:", report["date_start"], "to", report["date_end"])
print("Calibration quotes / returns:", len(calibration_quotes), "/", len(calibration_returns))
print("Evaluation quotes / returns:", len(evaluation_quotes), "/", len(evaluation_returns))
print("Source SHA-256:", report["source_sha256"])

returns[["previous_date", "date", "calendar_gap_days", "usd_lkr_rate", "previous_usd_lkr_rate", "log_return"]].head(10)